In [ ]:
df_bronze = spark.read\
    .csv("/Volumes/workspace/default/branchlens/branch_data.csv", header = True, inferSchema = True
         )

df_bronze.show(3)

In [ ]:
# finding number of rows and columns
print("Total number of rows: ",df_bronze.count())
print("Total number of columns: ",len(df_bronze.columns))
df_bronze.printSchema()

In [ ]:
# Checking count of null values
from pyspark.sql.functions import sum as spark_sum, col, cast

null_value_count = df_bronze.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns])
null_value_count.show(vertical = True)

In [ ]:
# Finding duplicates
total_count = df_bronze.count()
total_distinct_count = df_bronze.distinct().count()
print("Total count : ", total_count)
print("Total distinct count: ", total_distinct_count)
print("Number of Duplicate values: ", total_count - total_distinct_count)

In [ ]:
from pyspark.sql.functions import when, col
df_silver = df_bronze.withColumn("profit_status",when(col("profit_lakhs") > 0, "Profitable")\
    .otherwise("Loss")   
    )
df_silver.display()

In [ ]:
# Task 2 —
# Add another new column called performance_tier to df_silver based on profit_margin_pct —
# "Excellent" → profit_margin_pct >= 40
# "Good" → profit_margin_pct >= 25
# "Average" → profit_margin_pct >= 10
# "Poor" → anything below 10

df_silver = df_silver.withColumn("performance_tier", \
    when(col("profit_margin_pct") >= 40, "Excellent")\
    .when(col("profit_margin_pct") >= 25, "Good")\
    .when(col("profit_margin_pct") >= 10, "Average")\
    .otherwise("Poor")
    )
df_silver.display()

In [ ]:
#  Task 3 —
# Add one more column called npa_health to df_silver based on npa_ratio_pct —
# "Healthy" → npa_ratio_pct <= 1.5
# "Moderate" → npa_ratio_pct <= 3.5
# "Stressed" → anything above 3.5

df_silver = df_silver.withColumn("npa_health",\
    when(col("npa_ratio_pct") <= 1.5, "Healthy")\
    .when(col("npa_ratio_pct") <= 3.5, "Moderate")\
    .otherwise("Stressed")
    )
df_silver.display()

In [ ]:
print("Bronze Layer: ", len(df_bronze.columns))
print("Silver Layer: ", len(df_silver.columns))

In [ ]:
# Task 4 — First Gold table
# Create a dataframe called df_gold_branch_performance that shows for each branch_id and branch_name —
# Total revenue → sum of revenue_lakhs
# Total profit → sum of profit_lakhs
# Average profit margin → avg of profit_margin_pct
# Average customer satisfaction → avg of customer_satisfaction

from pyspark.sql.functions import sum, avg, col, round
df_gold_branch_performance = df_silver.groupBy("branch_id", "branch_name")\
    .agg(round(sum(col("revenue_lakhs")), 2).alias("total_revenue"),
         round(sum(col("profit_lakhs")), 2).alias("total_profit"),
         round(avg(col("profit_margin_pct")), 2).alias("average_profit_margin"),
         round(avg(col("customer_satisfaction")), 2).alias("average_customer_satisfaction")
         )
df_gold_branch_performance.display()

In [ ]:
# Task 5 —
# Sort this df_gold_branch_performance by total_revenue in descending order so the highest revenue branch appears first!

from pyspark.sql.functions import desc
df_gold_branch_performance = df_gold_branch_performance.orderBy(col("total_revenue").desc())
df_gold_branch_performance.display()

In [ ]:
#  Task 6 —
# Create a second Gold table called df_gold_region_summary that shows performance by region —
# Group by region and calculate —
# Total revenue → sum of revenue_lakhs
# Total profit → sum of profit_lakhs
# Average NPA ratio → avg of npa_ratio_pct
# Count of branches → use countDistinct("branch_id")

from pyspark.sql.functions import countDistinct
df_gold_region_summary = df_silver.groupBy(col("region"))\
    .agg(round(sum(col("revenue_lakhs")), 2).alias("total_revenue"),
         round(sum(col("profit_lakhs")), 2).alias("total_profit"),
         round(avg(col("npa_ratio_pct")), 2).alias("average_npa_ratio"),
         countDistinct(col("branch_id")).alias("count_of_branches")
         )
df_gold_region_summary.display()

In [ ]:
# Task 7 —
# Create one final Gold table called df_gold_anomaly_detection that finds stressed branches —
# Filter df_silver where ANY of these conditions are true —
# npa_health == "Stressed"
# performance_tier == "Poor"
# customer_satisfaction < 2.5

df_gold_anomaly_detection = df_silver.filter(
    (col("npa_health") == "Stressed")|
    (col("performance_tier") == "Poor")|
    (col("customer_satisfaction") < 3)
)

df_gold_anomaly_detection.display()
df_gold_anomaly_detection.count()

In [ ]:
# Check which branches appear most in anomalies
from pyspark.sql.functions import count

df_gold_anomaly_detection.groupBy("branch_id", "branch_name", ) \
    .agg(count("*").alias("anomaly_months")) \
    .orderBy(col("anomaly_months").desc()) \
    .display()

In [ ]:
df_gold_anomaly_detection \
    .filter(col("branch_id").isin(["BRN035", "BRN026", "BRN047"])) \
    .groupBy("branch_id", "branch_name") \
    .agg(count("*").alias("anomaly_months")) \
    .orderBy(col("anomaly_months").desc()) \
    .display()

In [ ]:
df_silver.filter(col("branch_id") == "BRN035") \
    .select("branch_id", "branch_name", "month", "year", 
            "month_num", "customer_satisfaction", 
            "npa_health", "performance_tier") \
    .orderBy("year", "month_num") \
    .display()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

windowSpec = Window.partitionBy("branch_id")
df_silver = df_silver.withColumn("branch_avg_csat", 
    round(avg("customer_satisfaction").over(windowSpec), 2))

df_silver.display()
print("Number of rows in silver layer is: ", len(df_silver.columns))

In [ ]:
df_silver = df_silver.withColumn("csat_anomaly", 
                                 when(col("customer_satisfaction") < col("branch_avg_csat") * 0.8, "Yes").otherwise("No"))
df_silver.display()
print(len(df_silver.columns))

In [ ]:
print("Number of csat_anomaly in positive: ",df_silver.filter(col('csat_anomaly') == 'Yes').count())

In [ ]:
df_silver.filter(col("csat_anomaly") == "Yes") \
    .select("branch_id", "branch_name", "month", 
            "customer_satisfaction", "branch_avg_csat") \
    .display()

In [ ]:
df_gold_anomaly_detection = df_silver.filter(
    (col("csat_anomaly") == "Yes") |
    (col("npa_health") == "Stressed") |
    (col("performance_tier") == "Poor")
)

print("Total anomaly records: ", df_gold_anomaly_detection.count())

In [ ]:
print("Bronze Layer rows: ", df_bronze.count())
print("Bronze Layer columns: ", len(df_bronze.columns))
print("Silver Layer rows: ", df_silver.count())
print("Silver Layer columns: ", len(df_silver.columns))
print("Gold - Branch Performance rows: ", df_gold_branch_performance.count())
print("Gold - Region Summary rows: ", df_gold_region_summary.count())
print("Gold - Anomaly Detection rows: ", df_gold_anomaly_detection.count())


In [ ]:
%pip install snowflake-connector-python

In [ ]:
# Snowflake connection options
sfOptions = {
    "host": "host_name",
    "sfuser": "user_name",
    "sfpassword": "password",
    "sfdatabase": "database_name",
    "sfschema": "schema_name",
    "sfwarehouse": "warehouse_name",
    "sfrole": "role"
}

In [ ]:
test_df = spark.read \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("query", "SELECT CURRENT_USER(), CURRENT_DATABASE()") \
    .load()

test_df.show()

In [ ]:
df_bronze.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "BRONZE_BRANCH_RAW") \
    .mode("overwrite") \
    .save()

In [ ]:
df_silver.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "SILVER_BRANCH") \
    .mode("overwrite") \
    .save()

df_gold_branch_performance.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "GOLD_BRANCH_PERFORMANCE") \
    .mode("overwrite") \
    .save()

df_gold_region_summary.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "GOLD_REGION_SUMMARY") \
    .mode("overwrite") \
    .save()

df_gold_anomaly_detection.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "GOLD_ANOMALY_DETECTION") \
    .mode("overwrite") \
    .save()